In [ ]:
import pandas as pd
from pathlib import Path

# Raiz do projeto: a pasta que contém UCMF_raw.csv. Procurar a partir da pasta atual
# para cima faz o notebook funcionar rodando da própria subpasta ou da raiz.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'UCMF_raw.csv').exists())

df = pd.read_csv(RAIZ / 'UCMF_raw.csv')

In [ ]:
display(df.head())

In [ ]:
display(df.describe())

In [ ]:
print(df.isnull().sum())

In [ ]:
# Converter 'Atendimento' para datetime, tratando erros e usando o formato correto se aplicável
df['Atendimento'] = pd.to_datetime(df['Atendimento'], errors='coerce', dayfirst=True)

# Converter 'DN' para datetime, tratando erros e usando o formato correto se aplicável
df['DN'] = pd.to_datetime(df['DN'], errors='coerce', dayfirst=True)

print("Colunas 'Atendimento' e 'DN' convertidas para datetime.")
print("Verificando os tipos de dados atualizados e as primeiras linhas:")

display(df.head())
df.info()

In [ ]:
import numpy as np

# Substituir 0 por NaN nas colunas 'Peso' e 'Altura'
df['Peso'] = df['Peso'].replace(0, np.nan)
df['Altura'] = df['Altura'].replace(0, np.nan)

# Substituir 'Não Calculado' por NaN na coluna 'PPA'
df['PPA'] = df['PPA'].replace('Não Calculado', np.nan)
df['PPA'] = df['PPA'].replace('#VALUE!', np.nan)

print("Valores '0' em 'Peso' e 'Altura', e 'Não Calculado' em 'PPA' foram substituídos por NaN.")

# Opcional: verificar a contagem de nulos novamente para essas colunas
print("\nNovas contagens de nulos para Peso, Altura e PPA:")
print(df[['Peso', 'Altura', 'PPA']].isnull().sum())

In [ ]:
# Remove a coluna 'Convenio'
df = df.drop(columns=['Convenio'])

print("Coluna 'Convenio' removida com sucesso.")

# Opcional: verificar as colunas restantes
print("\nNovas colunas no DataFrame:")
print(df.columns)

In [ ]:
# Calcula 'numero de episodios' como a contagem de atributos não nulos entre HDA 1 e HDA2
df['numero de episodios'] = df[['HDA 1', 'HDA2']].notnull().sum(axis=1)

# Preenche valores nulos em HDA 1 e HDA2 com strings vazias para facilitar a concatenação
df['HDA 1_temp'] = df['HDA 1'].fillna('')
df['HDA2_temp'] = df['HDA2'].fillna('')

# Concatena os sintomas. Se um for vazio, apenas o outro aparece. Remove espaços extras.
df['sintomas'] = df.apply(lambda row: ', '.join(filter(None, [row['HDA 1_temp'], row['HDA2_temp']])), axis=1)

# Se a coluna 'sintomas' estiver vazia (ambos HDA1 e HDA2 eram nulos), preenche com 'nao'
df['sintomas'] = df['sintomas'].replace('', 'nao')

# Remove as colunas temporárias
df = df.drop(columns=['HDA 1_temp', 'HDA2_temp'])

print("Colunas 'numero de episodios' e 'sintomas' criadas com sucesso.")
print("Verificando as novas colunas e as primeiras linhas:")
display(df[['HDA 1', 'HDA2', 'numero de episodios', 'sintomas']].head(10))


In [ ]:
df = df.drop(columns=['HDA 1', 'HDA2'])

In [ ]:
# Elimina objetos com idade nula ou negativa ou menor que 21
df = df[df['IDADE'].notnull() & (df['IDADE'] >= 0) & (df['IDADE'] <= 21)]

print("Objetos com idade nula ou negativa eliminados.")
print(f"Novo shape do DataFrame após eliminar idades nulas/negativas: {df.shape}")

# Elimina as colunas 'Atendimento' e 'DN'
df = df.drop(columns=['Atendimento', 'DN'])

print("Colunas 'Atendimento' e 'DN' eliminadas.")
print("Verificando as colunas restantes no DataFrame:")
print(df.columns)

In [ ]:
print(df.isnull().sum())

In [ ]:
print(f"Shape of DataFrame before dropping rows with 20 identical attributes: {df.shape}")

# Identify columns to consider for duplication (all except 'ID')
columns_for_duplicate_check = [col for col in df.columns if col != 'ID']

# Drop duplicate rows based on the selected columns, keeping the first occurrence
df_deduplicated = df.drop_duplicates(subset=columns_for_duplicate_check, keep='first')

print(f"Shape of DataFrame after dropping rows with 20 identical attributes: {df_deduplicated.shape}")

# Update the original DataFrame
df = df_deduplicated.copy()

In [ ]:
import numpy as np

# Preencher valores nulos em MOTIVO1 e MOTIVO2 com strings vazias para facilitar a concatenação
df['MOTIVO1_temp'] = df['MOTIVO1'].fillna('')
df['MOTIVO2_temp'] = df['MOTIVO2'].fillna('')

# Criar a nova coluna 'MOTIVOS' concatenando MOTIVO1 e MOTIVO2
# Usamos apply com lambda para lidar com a concatenação e evitar vírgulas extras se um estiver vazio
df['MOTIVOS'] = df.apply(lambda row: ', '.join(filter(None, [row['MOTIVO1_temp'], row['MOTIVO2_temp']])), axis=1)

# Remover as colunas temporárias e as colunas originais MOTIVO1 e MOTIVO2
df = df.drop(columns=['MOTIVO1_temp', 'MOTIVO2_temp', 'MOTIVO1', 'MOTIVO2'])

print("Colunas 'MOTIVO1' e 'MOTIVO2' foram combinadas em 'MOTIVOS'.")
print("Verificando as primeiras linhas da nova coluna 'MOTIVOS' e as colunas do DataFrame:")
display(df[['MOTIVOS']].head())
print(df.columns)

In [ ]:
print(f"Shape of DataFrame before dropping rows: {df.shape}")

# Count null values per row
null_counts_per_row = df.isnull().sum(axis=1)

# Filter out rows where 10 or more attributes are null
df_filtered = df[null_counts_per_row < 8]

print(f"Shape of DataFrame after dropping rows with >= 8 nulls: {df_filtered.shape}")

# You might want to update the original df variable
df = df_filtered.copy()

In [ ]:
print(df.isnull().sum())

In [ ]:
print(f"Shape do DataFrame antes de remover nulos em 'NORMAL X ANORMAL': {df.shape}")

# Remove as linhas onde 'NORMAL X ANORMAL' é nulo
df = df.dropna(subset=['NORMAL X ANORMAL'])

print(f"Shape do DataFrame após remover nulos em 'NORMAL X ANORMAL': {df.shape}")
print("Verificando a contagem de nulos para 'NORMAL X ANORMAL' após a remoção:")
print(df['NORMAL X ANORMAL'].isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calcular a contagem de nulos para CADA coluna (sem filtro)
null_counts = df.isnull().sum()

# Criar um DataFrame para facilitar a plotagem com seaborn
null_counts_df = null_counts.reset_index()
null_counts_df.columns = ['Variavel', 'Quantidade de Nulos']

# Ordenar as variáveis para melhor visualização (opcional)
null_counts_df = null_counts_df.sort_values(by='Quantidade de Nulos', ascending=False)

# Criar o gráfico de barras
plt.figure(figsize=(16, 8)) # Aumentar o tamanho para acomodar mais barras
sns.barplot(x='Variavel', y='Quantidade de Nulos', data=null_counts_df, palette='viridis', hue='Variavel', legend=False)
plt.title('Quantidade de Objetos com Atributo Nulo por Variável (Incluindo Zeros)')
plt.xlabel('Variável')
plt.ylabel('Quantidade de Nulos')
plt.xticks(rotation=60, ha='right') # Rotacionar rótulos para melhor legibilidade
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() # Ajustar o layout para evitar cortes nos rótulos
plt.show()

In [ ]:
df["PPA"].value_counts()

In [ ]:
print(df.isnull().sum())

In [ ]:
h = df["Altura"] / 100

# IMC a partir de peso e altura
m = df["IMC"].isna() & df["Peso"].notna() & df["Altura"].notna()
df.loc[m, "IMC"] = df.loc[m, "Peso"] / h[m]**2

# altura a partir de peso e IMC
m = df["Altura"].isna() & df["Peso"].notna() & df["IMC"].notna()
df.loc[m, "Altura"] = np.sqrt(df.loc[m, "Peso"] / df.loc[m, "IMC"]) * 100

# peso a partir de IMC e altura
m = df["Peso"].isna() & df["IMC"].notna() & df["Altura"].notna()
df.loc[m, "Peso"] = df.loc[m, "IMC"] * h[m]**2

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.shape)

In [ ]:
df[["Peso", "Altura", "IMC"]].isna().value_counts()

In [ ]:
# === Recalcular PPA a partir de PA, idade, sexo e altura (Fourth Report, 2004) ===

from scipy.stats import norm

COL_PAS, COL_PAD = "PA SISTOLICA", "PA DIASTOLICA"
COL_IDADE, COL_SEXO, COL_ALTURA, COL_PPA = "IDADE", "SEXO", "Altura", "PPA"
IDADE_EM = "anos"            # "anos" ou "meses"
ALTURA_EM = "cm"             # "cm" ou "m"
MAPA_SEXO = {"M": "M", "Masculino": "M", "F": "F", "Feminino": "F"}

# ---- Tabelas LMS do CDC (lidas direto do site; carregadas uma vez por sessão) ----
URL_CDC = "https://www.cdc.gov/growthcharts/data/zscore/"
if "LMS_CDC" not in globals():
    _tabs = [pd.read_csv(URL_CDC + "lenageinf.csv"), pd.read_csv(URL_CDC + "statage.csv")]
    _tabs[0] = _tabs[0][pd.to_numeric(_tabs[0]["Agemos"], errors="coerce") < 24]
    _lms = pd.concat(_tabs, ignore_index=True)
    _lms = _lms[pd.to_numeric(_lms["Sex"], errors="coerce").notna()]
    _lms = _lms.astype({"Sex": int, "Agemos": float, "L": float, "M": float, "S": float})
    LMS_CDC = {s: _lms[_lms["Sex"] == c].sort_values("Agemos") for s, c in (("M", 1), ("F", 2))}

# ---- Coeficientes do Fourth Report, Apêndice B (confira com a tabela publicada) ----
COEF = {
    ("M", "PAS"): (102.19768, [1.82416, 0.12776, 0.00249, -0.00135], [2.73157, -0.19618, -0.04659, 0.00947], 10.7128),
    ("M", "PAD"): (61.01217, [0.68314, -0.09835, 0.01711, 0.00045], [1.46993, -0.07849, -0.03144, 0.00967], 11.6032),
    ("F", "PAS"): (102.01027, [1.94397, 0.00598, -0.00789, -0.00059], [2.03526, 0.02534, -0.01884, 0.00121], 10.4855),
    ("F", "PAD"): (60.50510, [1.01301, 0.01157, 0.00424, -0.00137], [1.16641, 0.12795, -0.03869, -0.00079], 10.9573),
}
NOMES = {0: "Normal", 1: "Pré-hipertensão", 2: "HAS estágio 1", 3: "HAS estágio 2"}


def _z_altura(altura, meses, sexo):
    z = np.full(len(altura), np.nan)
    for s, t in LMS_CDC.items():
        m = (sexo == s) & ~np.isnan(altura) & (meses >= t["Agemos"].min()) & (meses <= t["Agemos"].max())
        L, M, S = (np.interp(meses[m], t["Agemos"], t[k]) for k in ("L", "M", "S"))
        L = np.where(np.abs(L) < 1e-6, 1e-6, L)   # evita divisão por zero quando L ≈ 0
        z[m] = ((altura[m] / M) ** L - 1) / (L * S)
    return z


def _classificar(pa, anos, sexo, zalt, comp, limite_adolesc):
    cls = np.full(len(pa), np.nan)
    for s in ("M", "F"):
        a, b, g, sd = COEF[(s, comp)]
        m = (sexo == s) & ~np.isnan(pa) & ~np.isnan(zalt) & (anos >= 1) & (anos < 18)
        y, z = anos[m] - 10, zalt[m]
        mu = a + sum(bj * y ** (j + 1) for j, bj in enumerate(b)) + sum(gk * z ** (k + 1) for k, gk in enumerate(g))
        p90, p95, p99 = (mu + norm.ppf(q) * sd for q in (0.90, 0.95, 0.99))
        v = pa[m]
        c = np.zeros(m.sum())
        c[(v >= p90) | (v >= limite_adolesc)] = 1
        c[v >= p95] = 2
        c[v > p99 + 5] = 3
        cls[m] = c
    return cls


_anos = df[COL_IDADE].to_numpy(float) / (12 if IDADE_EM == "meses" else 1)
_altura = df[COL_ALTURA].to_numpy(float) * (100 if ALTURA_EM == "m" else 1)
_sexo = df[COL_SEXO].map(MAPA_SEXO).to_numpy(object)
_pas, _pad = df[COL_PAS].to_numpy(float), df[COL_PAD].to_numpy(float)

_zalt = _z_altura(_altura, _anos * 12, _sexo)
_c_pas = _classificar(_pas, _anos, _sexo, _zalt, "PAS", 120)
_c_pad = _classificar(_pad, _anos, _sexo, _zalt, "PAD", 80)
_pior = np.where(np.isnan(_c_pas) | np.isnan(_c_pad), np.nan, np.fmax(_c_pas, _c_pad))

df["z_altura"] = _zalt
df["PPA_PAS_calc"] = pd.Series(_c_pas, index=df.index).map(NOMES)
df["PPA_PAD_calc"] = pd.Series(_c_pad, index=df.index).map(NOMES)
df["PPA_calc"] = pd.Series(_pior, index=df.index).map(NOMES)

# ---- Validação: PPA original x calculada, onde as duas existem ----
print("Rótulos originais da PPA:\n", df[COL_PPA].value_counts(dropna=False), "\n")
_ambos = df[COL_PPA].notna() & df["PPA_calc"].notna()
print(f"Registros com PPA original e calculada: {_ambos.sum()}")
print(f"PPA nula que pode ser preenchida: {(df[COL_PPA].isna() & df['PPA_calc'].notna()).sum()}\n")
display(pd.crosstab(df.loc[_ambos, COL_PPA], df.loc[_ambos, "PPA_calc"], margins=True))